In [ ]:
!pip install pandas scikit-image opencv-python numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 25.0 MB/s  0:00:00m0:00:010:01

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


### Tugas 1: Ekstraksi ciri dengan hasil tabel, boleh menggunakan metode apa saja

In [39]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

dataset_dir = "dataset"
folders = ["bacterial_spot", "healthy", "powdery", "virus"]

data_list = []

for label in folders:
    folder_path = os.path.join(dataset_dir, label)
    
    if not os.path.exists(folder_path):
        print(f"Folder {folder_path} tidak ditemukan, lewati...")
        continue

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path)
            
            if img is None:
                continue

            # Resize gambar
            img = cv2.resize(img, (128, 128))

            # 1. FITUR WARNA (RGB & HSV)
            mean_b, mean_g, mean_r = cv2.mean(img)[:3]
            std_b, std_g, std_r = np.std(img, axis=(0, 1))

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            mean_h, mean_s, mean_v = cv2.mean(hsv)[:3]

            # 2. FITUR TEKSTUR (GLCM)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)

            contrast = graycoprops(glcm, 'contrast')[0, 0]
            dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
            homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
            energy = graycoprops(glcm, 'energy')[0, 0]
            correlation = graycoprops(glcm, 'correlation')[0, 0]

            data_list.append({
                'filename': filename,
                'mean_r': mean_r,
                'mean_g': mean_g,
                'mean_b': mean_b,
                'std_r': std_r,
                'std_g': std_g,
                'std_b': std_b,
                'mean_h': mean_h,
                'mean_s': mean_s,
                'mean_v': mean_v,
                'glcm_contrast': contrast,
                'glcm_dissimilarity': dissimilarity,
                'glcm_homogeneity': homogeneity,
                'glcm_energy': energy,
                'glcm_correlation': correlation,
                'label': label
            })

df = pd.DataFrame(data_list)

df.to_csv("hasil_ekstraksi_ciri.csv", index=False)

print(f"Selesai! Total gambar yang diekstrak: {len(df)}")
df.head(150)

Selesai! Total gambar yang diekstrak: 280


,filename,mean_r,mean_g,mean_b,std_r,std_g,std_b,mean_h,mean_s,mean_v,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation,label
0,e8fa0ef6-3af0-4cb1-b22b-0fe913ec014a___UF.GRC_...,112.445984,118.152954,100.285645,26.179197,27.947791,32.240474,59.860107,36.578735,119.209106,46.703125,4.103346,0.266376,0.037410,0.964087,bacterial_spot
1,e76d1b61-beb7-448d-ab33-328d3e99cbbc___GCREC_B...,107.479797,105.381348,79.606079,32.433610,22.148587,36.396763,38.312561,83.798584,114.818054,88.766486,6.787525,0.155342,0.021663,0.927458,bacterial_spot
2,e82d7b9f-9e5a-4d2c-9094-4b3961a062bd___GCREC_B...,102.315735,107.621887,100.649109,42.714726,30.389862,37.622503,90.968872,43.214844,112.245056,70.564407,5.790785,0.192054,0.024874,0.970543,bacterial_spot
3,droppedImage_3_jpg.rf.943cc10e967c39f00e1088cd...,95.243103,145.081238,59.659119,44.901726,56.460176,39.666195,45.322205,148.728760,145.196289,328.100824,9.292630,0.197068,0.065094,0.932176,bacterial_spot
4,dec2db16-8bae-42e0-9b02-8e3d9f0793cc___GCREC_B...,93.502502,99.636963,87.838806,47.210716,32.973379,41.883224,40.735474,60.826416,104.878540,84.271900,6.207431,0.180339,0.021699,0.970662,bacterial_spot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,pm6_change_90.jpg,100.944641,145.522156,74.835266,45.218342,38.979543,42.718842,48.462708,133.608765,145.611328,194.594550,9.260642,0.137731,0.016204,0.940346,powdery
146,pm54_hight.jpg,145.339111,186.326294,122.782104,34.957251,36.844835,35.951125,50.005676,89.238525,186.405457,282.410925,11.864173,0.112689,0.015303,0.887302,powdery
147,pm59_change_90.jpg,132.904846,149.976318,119.376587,32.634011,30.272538,38.185415,48.252991,56.767151,150.063538,136.602116,7.824926,0.153803,0.018951,0.931085,powdery
148,pm25_change_180.jpg,85.180298,93.038879,68.099243,57.376373,55.910120,57.629472,58.609314,107.193359,97.484070,425.225640,11.587968,0.151560,0.015223,0.931252,powdery


### Tugas 2: Klasifikasi dengan menggunakan minimal 3 metode


#### Preprocessing

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# load data hasil
df = pd.read_csv("hasil_ekstraksi_ciri.csv")

# pisahkan Fitur dan Target Label
feature_cols = [
    'mean_r', 'mean_g', 'mean_b', 
    'std_r', 'std_g', 'std_b', 
    'mean_h', 'mean_s', 'mean_v',
    'glcm_contrast', 'glcm_dissimilarity', 
    'glcm_homogeneity', 'glcm_energy', 'glcm_correlation'
]

X = df[feature_cols].values
y = df['label'].values

# convert label string ke angka
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# bagi dataset training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# standardisasi fitur (untuk algoritma yang sensitif skala seperti SVM & KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

eval_results = {}

#### SVM

In [36]:
from sklearn.svm import SVC

# Inisialisasi & Training
model_svm = SVC(kernel="rbf", C=1.0, random_state=42)
model_svm.fit(X_train_scaled, y_train)

# Prediksi
y_pred_svm = model_svm.predict(X_test_scaled)

# Hitung Metrik Evaluasi
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred_svm, average="weighted"
)
acc = accuracy_score(y_test, y_pred_svm)

eval_results["SVM"] = {
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1,
}

# Tampilkan Classification Report sebagai Tabel Pandas
report_dict_svm = classification_report(
    y_test, y_pred_svm, target_names=le.classes_, output_dict=True
)
df_report_svm = pd.DataFrame(report_dict_svm).transpose().round(4)

df_report_svm

,precision,recall,f1-score,support
bacterial_spot,0.8000,0.8571,0.8276,14.0000
healthy,0.9333,1.0000,0.9655,14.0000
powdery,0.8333,0.7143,0.7692,14.0000
virus,1.0000,1.0000,1.0000,14.0000
accuracy,0.8929,0.8929,0.8929,0.8929
macro avg,0.8917,0.8929,0.8906,56.0000
weighted avg,0.8917,0.8929,0.8906,56.0000


#### Random Forest

In [35]:
from sklearn.ensemble import RandomForestClassifier

# Inisialisasi & Training
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

# Prediksi
y_pred_rf = model_rf.predict(X_test)

# Hitung Metrik Evaluasi
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred_rf, average="weighted"
)
acc = accuracy_score(y_test, y_pred_rf)

eval_results["Random Forest"] = {
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1,
}

# Tampilkan Classification Report sebagai Tabel Pandas
report_dict_rf = classification_report(
    y_test, y_pred_rf, target_names=le.classes_, output_dict=True
)
df_report_rf = pd.DataFrame(report_dict_rf).transpose().round(4)

df_report_rf

,precision,recall,f1-score,support
bacterial_spot,0.9231,0.8571,0.8889,14.0000
healthy,0.9333,1.0000,0.9655,14.0000
powdery,0.8571,0.8571,0.8571,14.0000
virus,1.0000,1.0000,1.0000,14.0000
accuracy,0.9286,0.9286,0.9286,0.9286
macro avg,0.9284,0.9286,0.9279,56.0000
weighted avg,0.9284,0.9286,0.9279,56.0000


#### KNN

In [34]:
from sklearn.neighbors import KNeighborsClassifier

# Inisialisasi & Training
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

# Prediksi
y_pred_knn = model_knn.predict(X_test_scaled)

# Hitung Metrik Evaluasi
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred_knn, average='weighted'
)
acc = accuracy_score(y_test, y_pred_knn)

eval_results['KNN'] = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1-Score': f1
}

report_dict = classification_report(y_test, y_pred_knn, target_names=le.classes_, output_dict=True)
df_report_knn = pd.DataFrame(report_dict).transpose().round(4)

df_report_knn

,precision,recall,f1-score,support
bacterial_spot,0.8000,0.8571,0.8276,14.000
healthy,0.8750,1.0000,0.9333,14.000
powdery,0.8182,0.6429,0.7200,14.000
virus,1.0000,1.0000,1.0000,14.000
accuracy,0.8750,0.8750,0.8750,0.875
macro avg,0.8733,0.8750,0.8702,56.000
weighted avg,0.8733,0.8750,0.8702,56.000


### Tugas 3: Evaluation matrix terdiri dari akurasi, recall, precision dan f1-score

In [38]:
df_summary = pd.DataFrame(eval_results).T
df_summary = df_summary[["Accuracy", "Precision", "Recall", "F1-Score"]]
df_summary = df_summary.round(4)

df_summary

,Accuracy,Precision,Recall,F1-Score
KNN,0.8750,0.8733,0.8750,0.8702
Random Forest,0.9286,0.9284,0.9286,0.9279
SVM,0.8929,0.8917,0.8929,0.8906
